In [0]:
# Initialize Silver tables with Liquid Clustering (using statistics-eligible columns)
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.fraud_lakehouse.silver_transactions (
    transaction_id STRING NOT NULL,
    account_id STRING NOT NULL,
    user_id STRING NOT NULL,
    amount_original DOUBLE,
    currency STRING,
    amount_usd DOUBLE,
    txn_type STRING,
    channel STRING,
    merchant_category STRING,
    location_country STRING,
    event_timestamp TIMESTAMP,
    is_suspicious_flag BOOLEAN,
    txns_last_1hr INT,
    spend_last_1hr_usd DOUBLE,
    prev_country STRING,
    is_impossible_travel BOOLEAN,
    composite_risk_score INT,
    risk_tier STRING,
    _silver_processed_at TIMESTAMP
)
USING DELTA
CLUSTER BY (account_id, event_timestamp)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.fraud_lakehouse.silver_account_profiles (
    account_sk STRING NOT NULL,
    account_id STRING NOT NULL,
    user_id STRING NOT NULL,
    primary_country STRING,
    risk_tier STRING,
    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN
)
USING DELTA
CLUSTER BY (account_id, effective_from)
""")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "fraud_lakehouse"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions"
SILVER_TXN_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"
SILVER_PROFILE_TABLE = f"{CATALOG}.{SCHEMA}.silver_account_profiles"
SILVER_CHECKPOINT = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints_volume/silver_transactions"

# Static FX Lookup Map (in production, joined against a daily FX rate dimension table)
FX_RATES_TO_USD = {
    "USD": 1.00,
    "EUR": 1.09,
    "GBP": 1.27,
    "CAD": 0.72,
    "INR": 0.012
}

def transform_and_upsert_silver(micro_batch_df, batch_id):
    if micro_batch_df.isEmpty():
        return

    # 1. Intra-batch Deduplication (keep latest ingested record per transaction_id)
    dedup_window = Window.partitionBy("transaction_id").orderBy(F.col("_ingest_timestamp").desc())
    clean_batch = (
        micro_batch_df
        .withColumn("_rn", F.row_number().over(dedup_window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
        # Standardize string fields
        .withColumn("currency", F.upper(F.trim(F.col("currency"))))
        .withColumn("channel", F.upper(F.trim(F.col("channel"))))
        .withColumn("txn_type", F.upper(F.trim(F.col("txn_type"))))
        .withColumn("location_country", F.upper(F.trim(F.col("location_country"))))
    )

    # 2. Normalize Currency to USD using Map Expression
    fx_map = F.create_map([F.lit(x) for pair in FX_RATES_TO_USD.items() for x in pair])
    normalized_batch = clean_batch.withColumn(
        "amount_usd",
        F.round(F.col("amount") * F.coalesce(fx_map[F.col("currency")], F.lit(1.0)), 2)
    ).withColumnRenamed("amount", "amount_original")

    # 3. Historical Context Lookback (last 1 hour of Silver data for active accounts in batch)
    # Ensures rolling window metrics remain accurate across micro-batch boundaries
    active_accounts = normalized_batch.select("account_id").distinct()
    min_batch_ts = normalized_batch.agg(F.min("event_timestamp")).collect()[0][0]

    historical_context = (
        spark.table(SILVER_TXN_TABLE)
        .join(active_accounts, on="account_id", how="inner")
        .filter(F.col("event_timestamp") >= (F.lit(min_batch_ts) - F.expr("INTERVAL 1 HOUR")))
        .select(
            "transaction_id", "account_id", "user_id", "amount_original", "currency",
            "amount_usd", "txn_type", "channel", "merchant_category",
            "location_country", "event_timestamp", "is_suspicious_flag"
        )
    )

    combined_window_df = (
        normalized_batch.select(historical_context.columns)
        .unionByName(historical_context)
        .dropDuplicates(["transaction_id"])
    )

    # 4. Feature Engineering: Rolling 1-Hour Velocity & Impossible Travel Detection
    # Range-based window over Unix epoch seconds (3600s = 1 hour)
    velocity_window = (
        Window.partitionBy("account_id")
        .orderBy(F.col("event_timestamp").cast("long"))
        .rangeBetween(-3600, 0)
    )
    sequence_window = Window.partitionBy("account_id").orderBy(F.col("event_timestamp").asc())

    enriched_df = (
        combined_window_df
        .withColumn("txns_last_1hr", F.count("transaction_id").over(velocity_window).cast("int"))
        .withColumn("spend_last_1hr_usd", F.round(F.sum("amount_usd").over(velocity_window), 2))
        .withColumn("prev_country", F.lag("location_country", 1).over(sequence_window))
        .withColumn("prev_timestamp", F.lag("event_timestamp", 1).over(sequence_window))
        # Flag impossible travel: country changed within 30 minutes (1800 seconds)
        .withColumn(
            "is_impossible_travel",
            F.when(
                F.col("prev_country").isNotNull() &
                (F.col("prev_country") != F.col("location_country")) &
                ((F.col("event_timestamp").cast("long") - F.col("prev_timestamp").cast("long")) <= 1800),
                F.lit(True)
            ).otherwise(F.lit(False))
        )
        # Compute deterministic Composite Fraud Risk Score (0 - 100)
        .withColumn(
            "composite_risk_score",
            F.least(
                F.lit(100),
                (
                    F.when(F.col("is_suspicious_flag"), 40).otherwise(0) +
                    F.when(F.col("amount_usd") > 10000, 25).when(F.col("amount_usd") > 5000, 10).otherwise(0) +
                    F.when(F.col("txns_last_1hr") >= 4, 20).otherwise(0) +
                    F.when(F.col("is_impossible_travel"), 35).otherwise(0) +
                    F.when(F.col("location_country").isin("KY", "PA", "VG"), 15).otherwise(0)
                )
            ).cast("int")
        )
        .withColumn(
            "risk_tier",
            F.when(F.col("composite_risk_score") >= 70, "CRITICAL")
             .when(F.col("composite_risk_score") >= 40, "HIGH")
             .when(F.col("composite_risk_score") >= 20, "MEDIUM")
             .otherwise("LOW")
        )
        .withColumn("_silver_processed_at", F.current_timestamp())
        .drop("prev_timestamp")
    )

    # Filter back to only the records belonging to the current micro-batch for upsert
    batch_ids = normalized_batch.select("transaction_id")
    final_upsert_df = enriched_df.join(batch_ids, on="transaction_id", how="inner")

    # 5. Idempotent Delta MERGE INTO (Upsert)
    silver_delta = DeltaTable.forName(spark, SILVER_TXN_TABLE)
    (
        silver_delta.alias("target")
        .merge(final_upsert_df.alias("source"), "target.transaction_id = source.transaction_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    # 6. SCD Type 2 Update for Account Risk Profiles
    update_scd2_profiles(final_upsert_df)


def update_scd2_profiles(batch_df):
    """Tracks historical shifts in an account's primary country or highest risk tier."""
    latest_account_state = (
        batch_df
        .withColumn(
            "_rn",
            F.row_number().over(Window.partitionBy("account_id").orderBy(F.col("event_timestamp").desc()))
        )
        .filter(F.col("_rn") == 1)
        .select(
            "account_id",
            "user_id",
            F.col("location_country").alias("primary_country"),
            "risk_tier",
            F.col("event_timestamp").alias("effective_from")
        )
    )

    profile_delta = DeltaTable.forName(spark, SILVER_PROFILE_TABLE)
    current_profiles = spark.table(SILVER_PROFILE_TABLE).filter(F.col("is_current") == True)

    # Identify records where risk_tier or primary_country actually changed
    changed_records = (
        latest_account_state.alias("src")
        .join(current_profiles.alias("tgt"), on="account_id", how="left")
        .filter(
            F.col("tgt.account_id").isNull() |
            (F.col("src.risk_tier") != F.col("tgt.risk_tier")) |
            (F.col("src.primary_country") != F.col("tgt.primary_country"))
        )
        .select("src.*")
    )

    if changed_records.isEmpty():
        return

    # Stage 1: Close out existing active records (set is_current = False, effective_to = new effective_from)
    (
        profile_delta.alias("target")
        .merge(
            changed_records.alias("source"),
            "target.account_id = source.account_id AND target.is_current = true"
        )
        .whenMatchedUpdate(set={
            "effective_to": "source.effective_from",
            "is_current": "false"
        })
        .execute()
    )

    # Stage 2: Insert new current versions
    new_versions = (
        changed_records
        .withColumn("account_sk", F.md5(F.concat_ws("||", F.col("account_id"), F.col("effective_from").cast("string"))))
        .withColumn("effective_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .select("account_sk", "account_id", "user_id", "primary_country", "risk_tier", "effective_from", "effective_to", "is_current")
    )

    new_versions.write.format("delta").mode("append").saveAsTable(SILVER_PROFILE_TABLE)


# Read incrementally from Bronze Delta table
bronze_stream = spark.readStream.format("delta").table(BRONZE_TABLE)

silver_query = (
    bronze_stream.writeStream
    .foreachBatch(transform_and_upsert_silver)
    .option("checkpointLocation", SILVER_CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)

silver_query.awaitTermination()
print("Silver CDC Merge, Velocity Enrichment, and SCD Type 2 updates completed.")

In [0]:
%sql
-- 1. Check Risk Tier distribution across deduplicated Silver transactions
SELECT 
    risk_tier,
    COUNT(*) AS txn_count,
    ROUND(AVG(amount_usd), 2) AS avg_amount_usd,
    SUM(CASE WHEN is_impossible_travel THEN 1 ELSE 0 END) AS impossible_travel_flags
FROM workspace.fraud_lakehouse.silver_transactions
GROUP BY risk_tier
ORDER BY txn_count DESC;